<h1>BUILDING WITH LANGCHAIN</h1>

<H4>BUILDING AN AGENT </H4>

In [14]:
!pip install langchain-google-genai

In [15]:
from langgraph.graph import StateGraph,START,END
from typing import Annotated
from dotenv import load_dotenv
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from IPython.display import Image, display
import os
import gradio as gr

In [16]:
load_dotenv( override=True)

True

<H4>CREATING A GRAPH</H4>

In [19]:
from langchain_groq import ChatGroq

In [33]:
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant"
)

# 2. Define the State type
from typing_extensions import TypedDict
from typing import List, Dict

class State(TypedDict):
    messages: List[Dict[str, str]]  # list of {"role": "user"/"assistant", "content": "..."}

# 3. Define chatbot node
def chatbot(state: State) -> State:
    # Always ensure there's at least one user message
    if not state.get("messages") or not state["messages"][-1]["content"].strip():
        return {"messages": state.get("messages", []) + [{"role": "assistant", "content": "⚠️ Please enter a message."}]}

    # Pass conversation to Groq
    response = llm.invoke(state["messages"])
    return {"messages": state["messages"] + [{"role": "assistant", "content": response.content}]}

# 4. Build graph
graph = StateGraph(State)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)
app = graph.compile()

# 5. Gradio chat interface
def chat(user_input: str, history: List[Dict[str, str]]):
    # Combine history + new user input
    messages = history + [{"role": "user", "content": user_input}]
    result = app.invoke({"messages": messages})
    return result["messages"][-1]["content"]

gr.ChatInterface(chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


<h4>Tavily AI</h4>

In [34]:
from tavily import TavilyClient

In [35]:
from langchain_community.tools.tavily_search import TavilySearchResults
search = TavilySearchResults(max_results=3)

# Run a query
results = search.run("Latest news about AI agents")
print(results)

tool = TavilySearchResults(max_results=3)
tools = [tool]

[{'title': 'AI Agents in 2025: Expectations vs. Reality - IBM', 'url': 'https://www.ibm.com/think/insights/ai-agents-2025-expectations-vs-reality', 'content': '“More and better agents” are on the way, predicts Time.1 “Autonomous ‘agents’ and profitability are likely to dominate the artificial intelligence agenda,” reports Reuters.2 “The age of agentic AI has arrived,” promises Forbes, in response to a claim from Nvidia’s Jensen Huang.3 [...] We spoke with several IBM experts to cut through the hype, with the goal of holding a more reasonable conversation about AI agents and what they’re going to do. Our team of informed insiders includes:\n\nMaryam Ashoori, PhD: Director of Product Management, IBM® watsonx.ai™\n\nMarina Danilevsky: Senior Research Scientist, Language Technologies\n\nVyoma Gajjar: AI Technical Solutions Architect\n\nChris Hay: Distinguished Engineer\n\nIndustry newsletter\n\n### The latest tech news, backed by expert insights [...] Stay updated about the new emerging AI

In [36]:
from langgraph.prebuilt import ToolNode,tools_condition

In [38]:
# 1. Setup Groq LLM
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant"
)
llm_bind_tools = llm.bind(tools)

# 2. Define the State type
from typing_extensions import TypedDict
from typing import List, Dict

class State(TypedDict):
    messages: List[Dict[str, str]]  # list of {"role": "user"/"assistant", "content": "..."}

# 3. Define chatbot node
def chatbot(state: State) -> State:
    # Always ensure there's at least one user message
    if not state.get("messages") or not state["messages"][-1]["content"].strip():
        return {"messages": state.get("messages", []) + [{"role": "assistant", "content": "⚠️ Please enter a message."}]}

    # Pass conversation to Groq
    response = llm_bind_tools.invoke(state["messages"])
    return {"messages": state["messages"] + [{"role": "assistant", "content": response.content}]}

# 4. Build graph
graph = StateGraph(State)
graph.add_node("chatbot", chatbot)
tool_node = ToolNode(tools= tools)
graph.add_node('tool',tool_node)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)
graph.add_conditional_edges('chatbot',tools_condition,'tools')
graph.add_edge('tool','chatbot')
app = graph.compile()

# 5. Gradio chat interface
def chat(user_input: str, history: List[Dict[str, str]]):
    # Combine history + new user input
    messages = history + [{"role": "user", "content": user_input}]
    result = app.invoke({"messages": messages})
    return result["messages"][-1]["content"]

gr.ChatInterface(chat, type="messages").launch()


TypeError: Runnable.bind() takes 1 positional argument but 2 were given

In [ ]:
from langchain.adapters.openai import convert_openai_messages

In [ ]:

from IPython.display import Markdown


prompt = [
    {
        'role': 'system',
        'content': f'''You are an AI critical thinker research assistant. 
        Your sole purpose is to write well written, objective and structured reports on given text.'''
    },
    {
        'role': 'user',
        'content': f'''Information: """{response}"""
        Using the above information, answer the following query: """{query}""" in a detailed report'''
    }
]

lc_messages = Markdown(prompt)
lc_messages

NameError: name 'response' is not defined

In [ ]:
from langgraph.graph import StateGraph
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

# --- new import
from langgraph.prebuilt import ToolNode, tools_condition

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

tool = TavilySearchResults(max_results=3)
tools = [tool]


# tell the LLM which tools it can call
llm_with_tools = llm.bind_tools(tools)

# change the chatbot() node function. Use llm_with_tools instead of llm.
def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)

# --- add this:
# run the tools if they are called by adding the tools to a new node.
# This node runs the tools requested in the last AIMessage.
tool_node = ToolNode(tools=[tool])
graph_builder.add_node("tools", tool_node)
# ---

# --- add this
# define the conditional_edges.
# we'll use the prebuilt tools_condition in the conditional_edge to route to the ToolNode if the last message has tool calls,
# otherwise, route to the end.
graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)
# ---

# any time a tool is called, we return to the chatbot to decide the next step
graph_builder.add_edge("tools", "chatbot")

graph_builder.set_entry_point("chatbot")

# we don't need to explicitly set a finish_point because our graph already has a way to finish!
# graph_builder.set_finish_point("chatbot")

graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
def chat(user_input: str, history):
    initial_state = State(messages=[{"role": "user", "content": user_input}])
    result = graph.invoke(initial_state)
    print(result)
    return result['messages'][-1].content


gr.ChatInterface(chat, type="messages").launch()